## Lab 8: Data warehousing

In [5]:
#Importing necessary packages
import duckdb
import pandas as pd

## Step 2

Use the data in the file PCPI24M1.csv to initialize your database. This file contains inflation information as available in January 2024.
Create a persistent database containing three tables: cpi_append, cpi_trunc and cpi_inc.

In [6]:
original_data = pd.read_csv("PCPI24M1.csv")
df = original_data
df["DATE"] = pd.to_datetime(df["DATE"], format="%Y:%m") #changing the date and time format
df

,DATE,CPI
0,1947-01-01,21.5
1,1947-02-01,21.6
2,1947-03-01,22.0
3,1947-04-01,22.0
4,1947-05-01,22.0
...,...,...
919,2023-08-01,306.3
920,2023-09-01,307.5
921,2023-10-01,307.6
922,2023-11-01,307.9


In [9]:
#creating a DuckDB database
file = "lab08.db"
con = duckdb.connect(file)

# adding the original data to the database
con.register("df", df)
con.execute("CREATE OR REPLACE TABLE original_data AS SELECT * FROM df")

# creating three working tables
con.execute("CREATE OR REPLACE TABLE cpi_append AS SELECT * FROM original_data")
con.execute("CREATE OR REPLACE TABLE cpi_trunc AS SELECT * FROM original_data")
con.execute("CREATE OR REPLACE TABLE cpi_inc AS SELECT * FROM original_data")

# show tables
con.execute("SHOW TABLES").fetchdf()

# close the connection
con.close()

## Step 3

Load additional inflation data contained in the file PCPI25M2.csv into your database. This file contains inflation data as available in February 2025. It contains additional observations and historical revisions with respect to the previous file.

In [4]:
new_data = pd.read_csv("PCPI25M2.csv")
new_data["DATE"] = pd.to_datetime(new_data["DATE"], format="%Y:%m")

### Append load method

In [11]:
def append_load(con, data):
    con.register("new_data", data)
    con.execute("INSERT INTO cpi_append SELECT * FROM new_data")

with duckdb.connect(file) as con:
    con.sql(
        "BEGIN TRANSACTION"
    )  # starting a transaction -- changes are synced once = improves performance
    append_load(con, new_data)
    con.sql("COMMIT")  # committing the transaction
    print(con.sql("SELECT * FROM cpi_append"))

┌─────────────────────┬────────┐
│        DATE         │  CPI   │
│      timestamp      │ double │
├─────────────────────┼────────┤
│ 1947-01-01 00:00:00 │   21.5 │
│ 1947-02-01 00:00:00 │   21.6 │
│ 1947-03-01 00:00:00 │   22.0 │
│ 1947-04-01 00:00:00 │   22.0 │
│ 1947-05-01 00:00:00 │   22.0 │
│ 1947-06-01 00:00:00 │   22.1 │
│ 1947-07-01 00:00:00 │   22.2 │
│ 1947-08-01 00:00:00 │   22.4 │
│ 1947-09-01 00:00:00 │   22.8 │
│ 1947-10-01 00:00:00 │   22.9 │
│          ·          │     ·  │
│          ·          │     ·  │
│          ·          │     ·  │
│ 2024-04-01 00:00:00 │  313.0 │
│ 2024-05-01 00:00:00 │  313.1 │
│ 2024-06-01 00:00:00 │  313.1 │
│ 2024-07-01 00:00:00 │  313.6 │
│ 2024-08-01 00:00:00 │  314.1 │
│ 2024-09-01 00:00:00 │  314.9 │
│ 2024-10-01 00:00:00 │  315.6 │
│ 2024-11-01 00:00:00 │  316.4 │
│ 2024-12-01 00:00:00 │  317.6 │
│ 2025-01-01 00:00:00 │  319.1 │
├─────────────────────┴────────┤
│ 1861 rows          2 columns │
│ (20 shown)                   │
└─────────